In [1]:
print('Hello, World!')

Hello, World!


In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

print(os.getenv('CHROMA_DIR'))
print(os.getenv('GOOGLE_API_KEY'))



None
None


In [4]:
from langchain_openai import ChatOpenAI

In [5]:
chat_llm = ChatOpenAI(model='gpt-4o-mini') #cheap
# chat_llm = ChatOpenAI(model='gpt-3.5-turbo') #cheapest

In [7]:
chat_llm.invoke('What is the capital of Bangladesh?').content


'The capital of Bangladesh is Dhaka.'

In [8]:
from typing_extensions import TypedDict, Annotated
import operator

from langchain_core.messages import AnyMessage, HumanMessage, AIMessage


class GraphState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

In [9]:
{
    "messages": [HumanMessage(content="Hi, this is Mir. Say hello in detail.")]
}

{'messages': [HumanMessage(content='Hi, this is Mir. Say hello in detail.', additional_kwargs={}, response_metadata={})]}

In [10]:
def llm_call(state: GraphState) -> dict:
    """Call the LLM using conversation messages and append AI response."""
    response = chat_llm.invoke(state["messages"])  # AIMessage
    return {
        "messages": [response]
    }

In [11]:
from langchain_core.messages import AIMessage

def token_counter(state: GraphState) -> dict:
    """Count tokens (simple word count) in the last AI message."""
    last_msg = state["messages"][-1]      # last message in the list
    text = last_msg.content              # message text
    token_number = len(text.split())     # count words (NOT real tokens)

    summary = f"Total token number in the generated answer (word count) is {token_number}"

    return {
        "messages": [AIMessage(content=summary)]
    }

In [12]:
from langgraph.graph import StateGraph

builder = StateGraph(GraphState)

builder.add_node("llm_call", llm_call)
builder.add_node("token_counter", token_counter)

In [13]:
builder.set_entry_point("llm_call")
builder.add_edge("llm_call", "token_counter")
builder.set_finish_point("token_counter")

In [12]:
app = builder.compile()

app.get_graph()

Graph(nodes={'__start__': Node(id='__start__', name='__start__', data=RunnableCallable(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None), 'llm_call': Node(id='llm_call', name='llm_call', data=llm_call(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None), 'token_counter': Node(id='token_counter', name='token_counter', data=token_counter(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None), '__end__': Node(id='__end__', name='__end__', data=None, metadata=None)}, edges=[Edge(source='__start__', target='llm_call', data=None, conditional=False), Edge(source='llm_call', target='token_counter', data=None, conditional=False), Edge(source='token_counter', target='__end__', data=None, conditional=False)])

In [17]:
from IPython.display import Image

In [19]:
display(Image(app.get_graph().draw_mermaid_png()))

NameError: name 'app' is not defined

In [20]:
result = app.invoke({
    "messages": [HumanMessage(content="Hi, this is Mir. Say hello in detail.")]
})

NameError: name 'app' is not defined

In [16]:
result

{'messages': [HumanMessage(content='Hi, this is Mir. Say hello in detail.', additional_kwargs={}, response_metadata={}),
  AIMessage(content="Hello Mir! It’s great to connect with you. I hope you're having a wonderful day so far. If there’s anything specific you’d like to discuss or any questions you have, feel free to share! Whether it’s about a topic of interest, something you’re curious about, or just a friendly chat, I’m here to engage with you in detail. What’s on your mind today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 80, 'prompt_tokens': 18, 'total_tokens': 98, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_373a14eb6f', 'id': 'chatcmpl-DBz1O6BGbhtDxCdX7cXyNqwKHgKaZ', 'service

{'messages': [HumanMessage(content='Hi, this is Mir. Say hello in detail.', additional_kwargs={}, response_metadata={}),
  AIMessage(content="Hello Mir! It’s great to connect with you. I hope you're having a wonderful day so far. If there’s anything specific you’d like to discuss or any questions you have, feel free to share! Whether it’s about a topic of interest, something you’re curious about, or just a friendly chat, I’m here to engage with you in detail. What’s on your mind today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 80, 'prompt_tokens': 18, 'total_tokens': 98, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_373a14eb6f', 'id': 'chatcmpl-DBz1O6BGbhtDxCdX7cXyNqwKHgKaZ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c8471-58da-70a2-a0e0-d0d417c66c6a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'output_tokens': 80, 'total_tokens': 98, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
  AIMessage(content='Total token number in the generated answer (word count) is 63', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}

In [17]:
for i, msg in enumerate(result["messages"], 1):
    print(f"\n--- Message {i} ({type(msg).__name__}) ---")
    print(msg.content)


--- Message 1 (HumanMessage) ---
Hi, this is Mir. Say hello in detail.

--- Message 2 (AIMessage) ---
Hello Mir! It’s great to connect with you. I hope you're having a wonderful day so far. If there’s anything specific you’d like to discuss or any questions you have, feel free to share! Whether it’s about a topic of interest, something you’re curious about, or just a friendly chat, I’m here to engage with you in detail. What’s on your mind today?

--- Message 3 (AIMessage) ---
Total token number in the generated answer (word count) is 63


In [20]:
result = app.invoke({
  "messages": [HumanMessage(content=
    "Hi, this is Mir. Write a friendly detailed greeting in 1-2 sentences. "
    "Include: (1) welcome line, (2) ask 2 questions, (3) offer help with AI + API testing."
  )]
})

In [ ]:
for i, msg in enumerate(result["messages"], 1):
    print(f"\n--- Message {i} ({type(msg).__name__}) ---")
    print(msg.content)